## 1. Imports and Paths

In [69]:
from pathlib import Path
import json
import os
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd

# Ajusta estas rutas a tu entorno actual

print("Current working directory:", os.getcwd())

#PROJECT_ROOT = Path("/home/victor/gw/cbc_pe")
#DATA_ROOT = Path("/home/victor/gw/cbc_pe/data")
PROJECT_ROOT = Path("/afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe")
DATA_ROOT = Path("/data/vserrano/cbc_pe_data")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"PROJECT_ROOT does not exist: {PROJECT_ROOT}")
else:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_ID = "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000"

CONFIG_DIR = PROJECT_ROOT / "configs" 
MODEL_DIR = DATA_ROOT / "models" / "checkpoints" / DATASET_ID
RESULTS_DIR = DATA_ROOT / "results" / DATASET_ID

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("CONFIG_DIR exists:", CONFIG_DIR.exists())
print("MODEL_DIR exists:", MODEL_DIR.exists())
print("RESULTS_DIR exists:", RESULTS_DIR.exists())

Current working directory: /afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe/notebooks
PROJECT_ROOT: /afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe
DATA_ROOT: /data/vserrano/cbc_pe_data
CONFIG_DIR exists: True
MODEL_DIR exists: True
RESULTS_DIR exists: True


In [70]:
generation_config_path =  PROJECT_ROOT / "configs" / "generation" / "generate_500k_bbh_4s.json"
training_config_path = PROJECT_ROOT / "configs" / "experiments" / "train_500k_M10_inputzscore_resdilated_emb64_d124_bs256_seed123.json"

with open(generation_config_path, "r") as f:
    gen_cfg = json.load(f)

with open(training_config_path, "r") as f:
    train_cfg = json.load(f)

print("Generation config keys:", gen_cfg.keys())
print("Training config keys:", train_cfg.keys())
print("Training parameters:", train_cfg["model"]["kwargs"].keys())

Generation config keys: dict_keys(['project_root', 'data_root', 'output', 'generation', 'simulation', 'parameter_sampler', 'detectors', 'signal_processor', 'label_transformer'])
Training config keys: dict_keys(['project_root', 'data_root', 'dataset', 'input_normalization', 'model', 'training', 'outputs'])
Training parameters: dict_keys(['n_detectors', 'n_outputs', 'embedding_dim', 'residual_channels', 'dilations', 'residual_kernel_size', 'dropout_conv', 'dropout_dense', 'num_groups'])


## 2. Metadata inspection

In [71]:
detectors = gen_cfg["detectors"]

fs = 4096
duration = gen_cfg["simulation"]["duration"]
n_samples = int(duration * fs)

context_start = gen_cfg["simulation"]["processing_context_start_samples"]
context_end = gen_cfg["simulation"]["processing_context_end_samples"]
processing_length = n_samples + context_start + context_end

signal_processor_cfg = gen_cfg["signal_processor"]

print("Detector order:", detectors)
print("Sampling frequency:", fs)
print("Final duration:", duration)
print("Final samples:", n_samples)
print("Processing context start samples:", context_start)
print("Processing context end samples:", context_end)
print("Processing input length:", processing_length)
print("Processing input duration:", processing_length / fs)

print("\nSignal processor:")
for k, v in signal_processor_cfg.items():
    print(f"  {k}: {v}")

Detector order: ['H1', 'L1', 'V1']
Sampling frequency: 4096
Final duration: 4.0
Final samples: 16384
Processing context start samples: 1664
Processing context end samples: 1664
Processing input length: 19712
Processing input duration: 4.8125

Signal processor:
  whitening_method: psd
  apply_highpass: True
  apply_lowpass: True
  apply_standardization: False
  output_mode: crop_to_config
  whitening_low_frequency_cutoff: 30.0
  whitening_max_filter_duration: 0.5
  whitening_trunc_method: hann
  highpass_frequency: 30.0
  lowpass_frequency: 512.0
  fir_order: 256
  fir_beta: 5.0
  remove_corrupted: True


In [72]:
### SANITY CHECKS to assure that the configuration parameters are consistent with the expected values

assert detectors == ["H1", "L1", "V1"], detectors
assert fs == 4096
assert n_samples == 16384
assert processing_length == 19712

assert signal_processor_cfg["whitening_method"] == "psd"
assert signal_processor_cfg["apply_highpass"] is True
assert signal_processor_cfg["apply_lowpass"] is True
assert signal_processor_cfg["apply_standardization"] is False
assert signal_processor_cfg["highpass_frequency"] == 30.0
assert signal_processor_cfg["lowpass_frequency"] == 512.0

print("Input contract validated.")

Input contract validated.


## 3. Model loading

In [73]:
from src.models.network import SimpleCNN_ResidualDilated

# Getting model info
model_cfg = train_cfg["model"]

print(model_cfg["class_name"])
print(model_cfg["kwargs"])

model = SimpleCNN_ResidualDilated(**model_cfg["kwargs"])
model.eval()

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f"Total parameters: {n_params:,}")
print(f"Trainable parameters: {n_trainable:,}")

SimpleCNN_ResidualDilated
{'n_detectors': 3, 'n_outputs': 3, 'embedding_dim': 64, 'residual_channels': 64, 'dilations': [1, 2, 4], 'residual_kernel_size': 7, 'dropout_conv': 0.05, 'dropout_dense': 0.1, 'num_groups': 8}
SimpleCNN_ResidualDilated(
  (block1): ConvBlock(
    (conv): Conv1d(3, 16, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 16, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (block2): ConvBlock(
    (conv): Conv1d(16, 32, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 32, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (block3): ConvBlock(
    (conv): Conv1d(32, 64, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 64, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=Fals

In [74]:
checkpoint_tag = train_cfg["outputs"]["checkpoint_tag"]
print("Checkpoint tag:", checkpoint_tag)

candidate_checkpoints = sorted(MODEL_DIR.rglob(f"*{checkpoint_tag}*"))
for p in candidate_checkpoints[:20]:
    print(p)

print("Number of candidates:", len(candidate_checkpoints))

Checkpoint tag: M10_inputzscore_resdilated_emb64_d124_train400k
/data/vserrano/cbc_pe_data/models/checkpoints/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_SimpleCNN_ResidualDilated_M10_inputzscore_resdilated_emb64_d124_train400k_MSELoss_seed123_checkpoint.pt
Number of candidates: 1


In [75]:
import sys
import torch

print("python:", sys.executable)
print("torch version:", torch.__version__)
print("torch path:", torch.__file__)
print("has torch._utils:", hasattr(torch, "_utils"))
print("cuda available:", torch.cuda.is_available())

python: /afs/ciemat.es/user/v/vserrano/miniconda3/envs/gw-env/bin/python
torch version: 2.11.0+cu130
torch path: /afs/ciemat.es/user/v/vserrano/miniconda3/envs/gw-env/lib/python3.10/site-packages/torch/__init__.py
has torch._utils: True
cuda available: False


In [76]:
checkpoint_path = candidate_checkpoints[-1]  # Load the last checkpoint
print(checkpoint_path)

ckpt = torch.load(checkpoint_path, map_location="cpu")
print(ckpt.keys())

model_config = ckpt["model_config"]
input_normalization_cfg = model_config.get("input_normalization", None)

print("input_normalization:", input_normalization_cfg)

model.load_state_dict(ckpt["model_state_dict"])
model.eval()

/data/vserrano/cbc_pe_data/models/checkpoints/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_SimpleCNN_ResidualDilated_M10_inputzscore_resdilated_emb64_d124_train400k_MSELoss_seed123_checkpoint.pt
dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'train_loss', 'best_val_loss', 'y_mean', 'y_std', 'model_config', 'training_config', 'elapsed_seconds', 'history'])
input_normalization: {'enabled': True, 'mode': 'per_sample_per_detector_zscore', 'eps': 1e-06}


SimpleCNN_ResidualDilated(
  (block1): ConvBlock(
    (conv): Conv1d(3, 16, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 16, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (block2): ConvBlock(
    (conv): Conv1d(16, 32, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 32, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (block3): ConvBlock(
    (conv): Conv1d(32, 64, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 64, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (residual_blocks): Sequential(
    (0): ResidualDilatedBlock(
      (conv1): Conv1d(64, 64, kernel_size=(7,), stride=(1,), padding=(3,))
      (norm1): GroupNorm(8, 64, eps=1e-05, affine=True)
      (activation

## 4. Label normalization statistics

In [77]:
# ------------------------------------------------------------
#  Label normalization statistics
# ------------------------------------------------------------

y_mean = np.asarray(ckpt["y_mean"], dtype=np.float64)
y_std = np.asarray(ckpt["y_std"], dtype=np.float64)

label_names = ["chirp_mass", "total_mass", "chi_eff"]

print("Label names:", label_names)
print("y_mean:", y_mean)
print("y_std:", y_std)

assert y_mean.shape == (3,)
assert y_std.shape == (3,)
assert np.all(np.isfinite(y_mean))
assert np.all(np.isfinite(y_std))
assert np.all(y_std > 0)

print("Label statistics validated.")

Label names: ['chirp_mass', 'total_mass', 'chi_eff']
y_mean: [3.74526215e+01 9.49739685e+01 1.02269521e-03]
y_std: [16.48472214 34.65016556  0.44085518]
Label statistics validated.


## 5. Helpers

In [78]:
def inverse_standardize(y_std_space):
    """
    Convert standardized labels/predictions to physical units:
    [chirp_mass, total_mass, chi_eff].
    """
    y_std_space = np.asarray(y_std_space, dtype=np.float64)
    return y_std_space * y_std + y_mean


def standardize(y_phys):
    """
    Convert physical labels to standardized training space.
    """
    y_phys = np.asarray(y_phys, dtype=np.float64)
    return (y_phys - y_mean) / y_std

test_std = np.zeros((1, 3))
test_phys = inverse_standardize(test_std)

print("Zero standardized corresponds to physical mean:")
for name, value in zip(label_names, test_phys[0]):
    print(f"{name}: {value:.6g}")

Zero standardized corresponds to physical mean:
chirp_mass: 37.4526
total_mass: 94.974
chi_eff: 0.0010227


In [ ]:
@torch.no_grad()
def predict_model_phys(model, X, y_mean, y_std):
    model.eval()

    device="cpu"

    xb = torch.from_numpy(X.astype(np.float32)).to(device)
    out = model(xb)

    if isinstance(out, tuple):
        pred_std, emb = out
    else:
        pred_std = out
        emb = None

    pred_std = pred_std.detach().cpu().numpy()
    pred_phys = pred_std * y_std[None, :] + y_mean[None, :]

    if emb is not None:
        emb = emb.detach().cpu().numpy()

    return pred_std, pred_phys, emb

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from src.models.evaluate import extract_predictions_and_embeddings

def predict_real_with_embeddings(model, X_real, device, batch_size=1):
    """
    Predict standardized labels and embeddings for real input.

    X_real shape:
        (N, C, T)

    Returns:
        pred_std: (N, D)
        pred_phys: (N, D)
        emb: (N, E)
    """
    X_tensor = torch.from_numpy(X_real.astype(np.float32))

    # Dummy y only because TensorDataset needs a second tensor for the evaluator.
    # It is not physically used for real-event inference.
    y_dummy = torch.zeros((X_tensor.shape[0], len(label_names)), dtype=torch.float32)

    loader = DataLoader(
        TensorDataset(X_tensor, y_dummy),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    pred_std, emb, y_eval_dummy = extract_predictions_and_embeddings(
        model=model,
        loader=loader,
        device=device,
    )

    pred_phys = pred_std * y_std[None, :] + y_mean[None, :]

    return pred_std, pred_phys, emb


In [80]:
def zscore_per_sample_per_detector_np(X, eps=1e-6):
    """
    X shape:
        (N, C, T) or (C, T)

    Applies per-sample, per-detector/channel z-score normalization.
    """
    X = np.asarray(X, dtype=np.float32)

    if X.ndim == 2:
        mean = X.mean(axis=1, keepdims=True)
        std = X.std(axis=1, keepdims=True)
        return ((X - mean) / (std + eps)).astype(np.float32)

    if X.ndim == 3:
        mean = X.mean(axis=2, keepdims=True)
        std = X.std(axis=2, keepdims=True)
        return ((X - mean) / (std + eps)).astype(np.float32)

    raise ValueError(f"Expected X shape (C,T) or (N,C,T), got {X.shape}")

## 6. Real-event data loading

We load long GWOSC strain files for one HLV event. Long files are needed because real-data PSD whitening requires an off-source PSD estimate. This replaces the synthetic training step `NoiseModel.get_psd(...)`.

In [81]:
from pathlib import Path
import urllib.request
import h5py
from pycbc.types import TimeSeries as PyCBCTimeSeries

event_name = "GW170814"
event_time = 1186741861.5

REAL_DETECTORS = ["H1", "L1", "V1"]

GWOSC_URLS_4096 = {
    "H1": "https://gwosc.org/archive/data/O2_4KHZ_R1/1185939456/H-H1_GWOSC_O2_4KHZ_R1-1186738176-4096.hdf5",
    "L1": "https://gwosc.org/archive/data/O2_4KHZ_R1/1185939456/L-L1_GWOSC_O2_4KHZ_R1-1186738176-4096.hdf5",
    "V1": "https://gwosc.org/archive/data/O2_4KHZ_R1/1185939456/V-V1_GWOSC_O2_4KHZ_R1-1186738176-4096.hdf5",
}

GWOSC_CACHE_DIR = Path("gwosc_cache")
GWOSC_CACHE_DIR.mkdir(exist_ok=True)

In [82]:
def download_if_needed(url, cache_dir=GWOSC_CACHE_DIR):
    filename = url.split("/")[-1]
    local_path = cache_dir / filename

    if not local_path.exists():
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, local_path)
    else:
        print(f"Using cached file: {filename}")

    return local_path


def read_gwosc_hdf5_as_pycbc_timeseries(path):
    with h5py.File(path, "r") as f:
        strain = f["strain"]["Strain"][:].astype(np.float64)
        gps_start = float(f["meta"]["GPSstart"][()])
        duration = float(f["meta"]["Duration"][()])
        delta_t = duration / len(strain)

    return PyCBCTimeSeries(
        strain,
        delta_t=delta_t,
        epoch=gps_start,
    )

In [60]:
raw_strains_4096 = {}

for ifo, url in GWOSC_URLS_4096.items():
    print(f"\nLoading {ifo}")

    local_path = download_if_needed(url)
    ts = read_gwosc_hdf5_as_pycbc_timeseries(local_path)

    raw_strains_4096[ifo] = ts

    print(
        ifo,
        "len:", len(ts),
        "duration:", float(ts.duration),
        "sample_rate:", float(ts.sample_rate),
        "start:", float(ts.start_time),
        "end:", float(ts.end_time),
    )

    assert abs(float(ts.sample_rate) - 4096.0) < 1e-6
    assert float(ts.start_time) <= event_time <= float(ts.end_time)


Loading H1
Using cached file: H-H1_GWOSC_O2_4KHZ_R1-1186738176-4096.hdf5
H1 len: 16777216 duration: 4096.0 sample_rate: 4096.0 start: 1186738176.0 end: 1186742272.0

Loading L1
Using cached file: L-L1_GWOSC_O2_4KHZ_R1-1186738176-4096.hdf5
L1 len: 16777216 duration: 4096.0 sample_rate: 4096.0 start: 1186738176.0 end: 1186742272.0

Loading V1
Using cached file: V-V1_GWOSC_O2_4KHZ_R1-1186738176-4096.hdf5
V1 len: 16777216 duration: 4096.0 sample_rate: 4096.0 start: 1186738176.0 end: 1186742272.0


## 7. Build real input with the training preprocessing contract

The simulated training pipeline generated noise and PSD from the same `NoiseModel`. For real data, the equivalent step is to estimate a detector-specific PSD from off-source real strain and pass it to the same `SignalProcessor`.

In [61]:
# ------------------------------------------------------------
# Build SignalProcessor with the same config as M08 training
# ------------------------------------------------------------

from src.config import SimulationConfig
from src.processing import SignalProcessor

# These values should already be defined from the generation config.
# We keep them explicit here to avoid hidden notebook-state errors.
fs = 4096
duration = 4.0
context_start = 1664
context_end = 1664

sim_config = SimulationConfig(
    duration=duration,
    processing_context_start_samples=context_start,
    processing_context_end_samples=context_end,
)

processor = SignalProcessor(
    config=sim_config,
    **gen_cfg["signal_processor"],
)

print("SignalProcessor ready")
print("whitening_method:", processor.whitening_method)
print("apply_highpass:", processor.apply_highpass)
print("apply_lowpass:", processor.apply_lowpass)
print("apply_standardization:", processor.apply_standardization)
print("output_mode:", processor.output_mode)
print("highpass_frequency:", processor.highpass_frequency)
print("lowpass_frequency:", processor.lowpass_frequency)
print("processing_length:", sim_config.processing_length)
print("output_length:", sim_config.length)

assert processor.whitening_method == "psd"
assert processor.apply_highpass is True
assert processor.apply_lowpass is True
assert processor.apply_standardization is False
assert processor.output_mode == "crop_to_config"
assert sim_config.processing_length == 19712
assert sim_config.length == 16384

SignalProcessor ready
whitening_method: psd
apply_highpass: True
apply_lowpass: True
apply_standardization: False
output_mode: crop_to_config
highpass_frequency: 30.0
lowpass_frequency: 512.0
processing_length: 19712
output_length: 16384


In [62]:
from pycbc.psd import interpolate, inverse_spectrum_truncation

final_duration = float(duration)

final_length = int(final_duration * fs)
processing_length = final_length + context_start + context_end

processing_duration = processing_length / fs
processing_delta_f = 1.0 / processing_duration
processing_flength = processing_length // 2 + 1

print("final_length:", final_length)
print("processing_length:", processing_length)
print("processing_duration:", processing_duration)
print("processing_delta_f:", processing_delta_f)
print("processing_flength:", processing_flength)

assert final_length == 16384
assert processing_length == 19712

final_length: 16384
processing_length: 19712
processing_duration: 4.8125
processing_delta_f: 0.2077922077922078
processing_flength: 9857


In [63]:
def estimate_offsource_psd_long(
    strain,
    event_time,
    psd_start_offset=-512.0,
    psd_end_offset=-128.0,
    psd_segment_duration=8.0,
    low_frequency_cutoff=30.0,
    max_filter_duration=0.5,
):
    psd_start = event_time + psd_start_offset
    psd_end = event_time + psd_end_offset

    available_start = float(strain.start_time)
    available_end = float(strain.end_time)

    if psd_start < available_start or psd_end > available_end:
        raise ValueError(
            f"PSD window [{psd_start}, {psd_end}] outside available "
            f"[{available_start}, {available_end}]"
        )

    psd_data = strain.time_slice(psd_start, psd_end)

    psd = psd_data.psd(psd_segment_duration)
    psd = interpolate(psd, processing_delta_f)

    max_filter_len = int(round(max_filter_duration * fs))

    psd = inverse_spectrum_truncation(
        psd,
        max_filter_len=max_filter_len,
        low_frequency_cutoff=low_frequency_cutoff,
        trunc_method="hann",
    )

    if len(psd) > processing_flength:
        psd = psd[:processing_flength]
    elif len(psd) < processing_flength:
        raise ValueError(f"PSD too short: {len(psd)} < {processing_flength}")

    if not np.all(np.isfinite(psd.numpy())):
        raise ValueError("PSD contains non-finite values.")

    return psd

In [64]:
def build_real_input_like_training(
    *,
    raw_strains_long,
    center_time,
    processor,
    psd_start_offset=-512.0,
    psd_end_offset=-128.0,
    psd_segment_duration=8.0,
):
    """
    Build one real HLV input using the same preprocessing contract as M08 training.

    The final 4 s output window is centered at `center_time`.
    The input to SignalProcessor includes the same processing context used in training.
    """
    output_start = center_time - final_duration / 2
    output_end = center_time + final_duration / 2

    processing_start = output_start - context_start / fs
    processing_end = output_end + context_end / fs

    real_segments = {}

    for ifo in REAL_DETECTORS:
        seg = raw_strains_long[ifo].time_slice(processing_start, processing_end)

        if len(seg) != processing_length:
            raise ValueError(
                f"{ifo}: expected {processing_length}, got {len(seg)}"
            )

        real_segments[ifo] = seg

    psds = {}

    for ifo in REAL_DETECTORS:
        psds[ifo] = estimate_offsource_psd_long(
            raw_strains_long[ifo],
            event_time=center_time,
            psd_start_offset=psd_start_offset,
            psd_end_offset=psd_end_offset,
            psd_segment_duration=psd_segment_duration,
            low_frequency_cutoff=30.0,
            max_filter_duration=0.5,
        )

    processed = processor.process_network(
        strains=real_segments,
        psds=psds,
    )

    X = np.stack(
        [processed[ifo].numpy() for ifo in REAL_DETECTORS],
        axis=0,
    ).astype(np.float32)

    X = X[None, :, :]

    if X.shape != (1, 3, 16384):
        raise ValueError(f"Bad X shape: {X.shape}")

    if not np.all(np.isfinite(X)):
        raise ValueError("X contains non-finite values.")

    return X, processed, psds, real_segments

In [65]:
def summarize_real_scale(X, tag):
    rows = []

    for j, ifo in enumerate(REAL_DETECTORS):
        arr = X[0, j]

        rows.append({
            "preprocessing": tag,
            "detector": ifo,
            "real_mean": float(np.mean(arr)),
            "real_std": float(np.std(arr)),
            "real_maxabs": float(np.max(np.abs(arr))),
            "finite": bool(np.all(np.isfinite(arr))),
        })

    return pd.DataFrame(rows)

## 8. Point prediction

In [66]:
X_real, processed_real, psds_real, real_segments = build_real_input_like_training(
    raw_strains_long=raw_strains_4096,
    center_time=event_time,
    processor=processor,
)

In [67]:
X_real_raw = X_real.copy()

X_real = zscore_per_sample_per_detector_np(
    X_real_raw,
    eps=input_normalization_cfg.get("eps", 1e-6),
)

print("Raw real scale:")
display(summarize_real_scale(X_real_raw, tag="raw_processed_real"))

print("Z-scored real scale:")
display(summarize_real_scale(X_real, tag="zscore_processed_real"))

Raw real scale:


,preprocessing,detector,real_mean,real_std,real_maxabs,finite
0,raw_processed_real,H1,0.001536,23.639956,105.104576,True
1,raw_processed_real,L1,-0.012151,86.263588,206.907181,True
2,raw_processed_real,V1,0.023862,154.776749,473.826996,True


Z-scored real scale:


,preprocessing,detector,real_mean,real_std,real_maxabs,finite
0,zscore_processed_real,H1,-1.650187e-08,1.0,4.445991,True
1,zscore_processed_real,L1,6.228220e-09,1.0,2.398405,True
2,zscore_processed_real,V1,-1.117587e-08,1.0,3.061512,True


In [68]:
print("X_real shape:", X_real.shape)
print("finite:", np.all(np.isfinite(X_real)))
print("mean:", float(X_real.mean()))
print("std:", float(X_real.std()))

assert X_real.shape == (1, 3, 16384)
assert np.all(np.isfinite(X_real))

X_real shape: (1, 3, 16384)
finite: True
mean: -7.1498411635673165e-09
std: 0.9999999403953552


In [84]:
pred_real_std, pred_real_phys, emb_real = predict_real_with_embeddings(
    model=model,
    X_real=X_real,
    device="cpu",
    batch_size=1,
)

print("pred_real_std:", pred_real_std.shape)
print("pred_real_phys:", pred_real_phys.shape)
print("emb_real:", None if emb_real is None else emb_real.shape)

real_m10_point_df = pd.DataFrame({
    "event": [event_name] * len(label_names),
    "label": label_names,
    "pred_std": pred_real_std[0],
    "pred": pred_real_phys[0],
})

display(real_m10_point_df)

pred_real_std: (1, 3)
pred_real_phys: (1, 3)
emb_real: (1, 64)


,event,label,pred_std,pred
0,GW170814,chirp_mass,-0.732548,25.376774
1,GW170814,total_mass,-1.000442,60.308495
2,GW170814,chi_eff,-0.480637,-0.210869


## 9. Real/synthetic scale diagnostic

In [85]:
import h5py

RUN_ENV = "local"  # "local" or "vm"

if RUN_ENV == "vm":
    DATA_ROOT = Path("/data/vserrano/cbc_pe/data")
    PROJECT_ROOT = Path("/data/vserrano/gw/Gravitational-Waves-Lab/cbc_pe")
else:
    DATA_ROOT = Path("/data/vserrano/cbc_pe_data")
    PROJECT_ROOT = Path("/afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe")

dataset_path = DATA_ROOT / "processed" / "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000" / "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000.h5"

rng = np.random.default_rng(123)
n_probe = 5000

with h5py.File(dataset_path, "r") as f:
    n_total = f["X"].shape[0]
    probe_idx = np.sort(rng.choice(n_total, size=n_probe, replace=False))
    X_probe = f["X"][probe_idx].astype(np.float32)

rows = []

for j, ifo in enumerate(REAL_DETECTORS):
    channel_std = X_probe[:, j, :].std(axis=1)
    channel_mean = X_probe[:, j, :].mean(axis=1)
    channel_maxabs = np.max(np.abs(X_probe[:, j, :]), axis=1)

    rows.append({
        "detector": ifo,
        "train_mean_of_means": float(np.mean(channel_mean)),
        "train_median_std": float(np.median(channel_std)),
        "train_q05_std": float(np.quantile(channel_std, 0.05)),
        "train_q95_std": float(np.quantile(channel_std, 0.95)),
        "train_median_maxabs": float(np.median(channel_maxabs)),
        "train_q95_maxabs": float(np.quantile(channel_maxabs, 0.95)),
    })

train_scale_df = pd.DataFrame(rows)
display(train_scale_df)

,detector,train_mean_of_means,train_median_std,train_q05_std,train_q95_std,train_median_maxabs,train_q95_maxabs
0,H1,-0.000224,22.066248,21.580492,22.624434,92.406082,122.905823
1,L1,-0.000067,22.060675,21.584286,22.673035,92.248581,124.518295
2,V1,0.000006,22.005814,21.533955,22.671404,91.671600,129.735229


In [86]:
real_scale_df = summarize_real_scale(
    X_real,
    tag="offsource_psd_4096s_centered",
)

scale_comparison_df = real_scale_df.merge(
    train_scale_df,
    on="detector",
    how="left",
)

scale_comparison_df["std_ratio_real_to_train_median"] = (
    scale_comparison_df["real_std"] / scale_comparison_df["train_median_std"]
)

scale_comparison_df["maxabs_ratio_real_to_train_q95"] = (
    scale_comparison_df["real_maxabs"] / scale_comparison_df["train_q95_maxabs"]
)

display(scale_comparison_df)

,preprocessing,detector,real_mean,real_std,real_maxabs,finite,train_mean_of_means,train_median_std,train_q05_std,train_q95_std,train_median_maxabs,train_q95_maxabs,std_ratio_real_to_train_median,maxabs_ratio_real_to_train_q95
0,offsource_psd_4096s_centered,H1,-1.650187e-08,1.0,4.445991,True,-0.000224,22.066248,21.580492,22.624434,92.406082,122.905823,0.045318,0.036174
1,offsource_psd_4096s_centered,L1,6.228220e-09,1.0,2.398405,True,-0.000067,22.060675,21.584286,22.673035,92.248581,124.518295,0.045330,0.019261
2,offsource_psd_4096s_centered,V1,-1.117587e-08,1.0,3.061512,True,0.000006,22.005814,21.533955,22.671404,91.671600,129.735229,0.045443,0.023598


The intervals are calibrated on synthetic data. Therefore, for real data they should be interpreted as a transfer diagnostic unless the real input distribution is shown to match the synthetic calibration distribution sufficiently well.

## 10. PSD-window sensitivity

In [93]:
PSD_WINDOWS = [
    (-1600.0, -1216.0),
    (-1400.0, -1016.0),
    (-1200.0, -816.0),
    (-1024.0, -640.0),
    (-768.0, -384.0),
    (-512.0, -128.0),
]

psd_sensitivity_results_m10 = []
psd_sensitivity_scales_m10 = []

for psd_start_offset, psd_end_offset in PSD_WINDOWS:
    tag = f"psd_{int(psd_start_offset)}_{int(psd_end_offset)}"

    print("\n", "=" * 80)
    print(tag)

    X_tmp_raw, processed_tmp, psds_tmp, segs_tmp = build_real_input_like_training(
        raw_strains_long=raw_strains_4096,
        center_time=event_time,
        processor=processor,
        psd_start_offset=psd_start_offset,
        psd_end_offset=psd_end_offset,
        psd_segment_duration=8.0,
    )

    scale_raw_tmp = summarize_real_scale(X_tmp_raw, tag=tag + "_raw")

    X_tmp = zscore_per_sample_per_detector_np(
        X_tmp_raw,
        eps=input_normalization_cfg.get("eps", 1e-6),
    )

    scale_z_tmp = summarize_real_scale(X_tmp, tag=tag + "_zscore")

    pred_std, pred_phys, emb = predict_model_phys(
        model=model,
        X=X_tmp,
        y_mean=y_mean,
        y_std=y_std,
    )

    for j, name in enumerate(label_names):
        psd_sensitivity_results_m10.append({
            "preprocessing": tag,
            "label": name,
            "pred": float(pred_phys[0, j]),
        })

    psd_sensitivity_scales_m10.append(scale_raw_tmp)
    psd_sensitivity_scales_m10.append(scale_z_tmp)

psd_sensitivity_result_m10_df = pd.DataFrame(psd_sensitivity_results_m10)
psd_sensitivity_scale_m10_df = pd.concat(psd_sensitivity_scales_m10, ignore_index=True)

display(
    psd_sensitivity_result_m10_df.pivot_table(
        index="preprocessing",
        columns="label",
        values="pred",
        aggfunc="first",
    )
)


psd_-1600_-1216

psd_-1400_-1016

psd_-1200_-816

psd_-1024_-640

psd_-768_-384

psd_-512_-128


label,chi_eff,chirp_mass,total_mass
preprocessing,,,
psd_-1024_-640,-0.205554,25.418858,60.432048
psd_-1200_-816,-0.209729,25.405510,60.373589
psd_-1400_-1016,-0.210981,25.419274,60.356674
psd_-1600_-1216,-0.217462,25.408201,60.231227
psd_-512_-128,-0.210869,25.376774,60.308495
psd_-768_-384,-0.205483,25.385564,60.361507


In [95]:
display(
    psd_sensitivity_result_m10_df.pivot_table(
        index="preprocessing",
        columns="label",
        values="pred",
        aggfunc="first",
    )
)

display(
    psd_sensitivity_scale_m10_df.pivot_table(
        index="preprocessing",
        columns="detector",
        values="real_std",
        aggfunc="first",
    )
)

label,chi_eff,chirp_mass,total_mass
preprocessing,,,
psd_-1024_-640,-0.205554,25.418858,60.432048
psd_-1200_-816,-0.209729,25.405510,60.373589
psd_-1400_-1016,-0.210981,25.419274,60.356674
psd_-1600_-1216,-0.217462,25.408201,60.231227
psd_-512_-128,-0.210869,25.376774,60.308495
psd_-768_-384,-0.205483,25.385564,60.361507


detector,H1,L1,V1
preprocessing,,,
psd_-1024_-640_raw,23.615892,86.220123,156.278931
psd_-1024_-640_zscore,1.000000,1.000000,1.000000
psd_-1200_-816_raw,23.519014,84.854294,160.171844
psd_-1200_-816_zscore,1.000000,1.000000,1.000000
psd_-1400_-1016_raw,23.486172,84.679619,159.461594
psd_-1400_-1016_zscore,1.000000,1.000000,1.000000
psd_-1600_-1216_raw,23.399437,85.420967,153.389740
psd_-1600_-1216_zscore,1.000000,1.000000,1.000000
psd_-512_-128_raw,23.639956,86.263588,154.776749


The M10 prediction is stable under reasonable off-source PSD-window choices and remains in a physically plausible BBH region. This indicates that, after applying the training-compatible per-sample/per-detector z-score normalization, the real-event prediction is not driven by a particular PSD-window choice.

## 11. Center-time sensitivity

In [96]:
CENTER_OFFSETS = np.array([
    -1.50, -1.25, -1.00, -0.75, -0.50, -0.25,
     0.00,
     0.25,  0.50,  0.75,  1.00,  1.25,  1.50,
])

time_sensitivity_results_m10 = []
time_sensitivity_scales_m10 = []

for dt_center in CENTER_OFFSETS:
    center_time = event_time + float(dt_center)
    tag = f"center_{dt_center:+.2f}s"

    print("\n", "=" * 80)
    print(tag)

    X_tmp_raw, processed_tmp, psds_tmp, segs_tmp = build_real_input_like_training(
        raw_strains_long=raw_strains_4096,
        center_time=center_time,
        processor=processor,
        psd_start_offset=-512.0,
        psd_end_offset=-128.0,
        psd_segment_duration=8.0,
    )

    scale_raw_tmp = summarize_real_scale(X_tmp_raw, tag=tag + "_raw")
    scale_raw_tmp["center_offset_s"] = float(dt_center)
    scale_raw_tmp["scale_type"] = "raw"

    X_tmp = zscore_per_sample_per_detector_np(
        X_tmp_raw,
        eps=input_normalization_cfg.get("eps", 1e-6),
    )

    scale_z_tmp = summarize_real_scale(X_tmp, tag=tag + "_zscore")
    scale_z_tmp["center_offset_s"] = float(dt_center)
    scale_z_tmp["scale_type"] = "zscore"

    pred_std, pred_phys, emb = predict_model_phys(
        model=model,
        X=X_tmp,
        y_mean=y_mean,
        y_std=y_std,
    )

    for j, name in enumerate(label_names):
        time_sensitivity_results_m10.append({
            "center_offset_s": float(dt_center),
            "preprocessing": tag,
            "label": name,
            "pred": float(pred_phys[0, j]),
        })

    time_sensitivity_scales_m10.append(scale_raw_tmp)
    time_sensitivity_scales_m10.append(scale_z_tmp)

time_sensitivity_result_m10_df = pd.DataFrame(time_sensitivity_results_m10)
time_sensitivity_scale_m10_df = pd.concat(time_sensitivity_scales_m10, ignore_index=True)



center_-1.50s

center_-1.25s

center_-1.00s

center_-0.75s

center_-0.50s

center_-0.25s

center_+0.00s

center_+0.25s

center_+0.50s

center_+0.75s

center_+1.00s

center_+1.25s

center_+1.50s


In [97]:
display(
    time_sensitivity_result_m10_df.pivot_table(
        index="center_offset_s",
        columns="label",
        values="pred",
        aggfunc="first",
    )
)

display(
    time_sensitivity_scale_m10_df.pivot_table(
        index=["scale_type", "center_offset_s"],
        columns="detector",
        values="real_std",
        aggfunc="first",
    )
)

label,chi_eff,chirp_mass,total_mass
center_offset_s,,,
-1.50,-0.200654,24.969670,59.705730
-1.25,-0.198364,24.827255,59.436818
-1.00,-0.194265,25.024147,59.799953
-0.75,-0.198551,25.111650,59.895172
-0.50,-0.212079,25.031039,59.736619
-0.25,-0.182428,25.574158,60.855034
0.00,-0.210869,25.376774,60.308495
0.25,-0.206188,25.391656,60.587415
0.50,-0.230311,25.375505,60.435036


detector                           H1         L1          V1
scale_type center_offset_s                                  
raw        -1.50            23.750530  85.276680  155.085251
           -1.25            23.642326  85.589439  153.406326
           -1.00            23.566851  85.777832  152.741684
           -0.75            23.599737  87.280190  151.946182
           -0.50            23.615847  86.769058  152.569275
           -0.25            23.582626  86.739029  153.733734
            0.00            23.639956  86.263588  154.776749
            0.25            23.574713  86.460243  153.467804
            0.50            23.482624  86.824142  152.618958
            0.75            23.256563  86.968338  153.396851
            1.00            23.106743  86.560318  154.531921
            1.25            23.132301  85.898438  154.786179
            1.50            23.163891  84.174126  155.102814
zscore     -1.50             1.000000   1.000000    1.000000
           -1.25             1.000000   1.000000    1.000000
           -1.00             1.000000   1.000000    1.000000
           -0.75             1.000000   1.000000    1.000000
           -0.50             1.000000   1.000000    1.000000
           -0.25             1.000000   1.000000    1.000000
            0.00             1.000000   1.000000    1.000000
            0.25             1.000000   1.000000    1.000000
            0.50             1.000000   1.000000    1.000000
            0.75             1.000000   1.000000    1.000000
            1.00             1.000000   1.000000    1.000000
            1.25             1.000000   1.000000    1.000000
            1.50             1.000000   1.000000    1.000000

## 12. Off-source real-noise controls

In [98]:
NOISE_CENTER_OFFSETS = np.array([
    -1800.0, -1600.0, -1400.0, -1200.0,
    -1000.0, -800.0, -600.0, -400.0,
    200.0, 300.0,
])

noise_control_results_m10 = []
noise_control_scales_m10 = []

for dt_center in NOISE_CENTER_OFFSETS:
    center_time = event_time + float(dt_center)
    tag = f"noise_center_{dt_center:+.0f}s"

    print("\n", "=" * 80)
    print(tag)

    X_tmp_raw, processed_tmp, psds_tmp, segs_tmp = build_real_input_like_training(
        raw_strains_long=raw_strains_4096,
        center_time=center_time,
        processor=processor,
        psd_start_offset=-512.0,
        psd_end_offset=-128.0,
        psd_segment_duration=8.0,
    )

    scale_raw_tmp = summarize_real_scale(X_tmp_raw, tag=tag + "_raw")

    X_tmp = zscore_per_sample_per_detector_np(
        X_tmp_raw,
        eps=input_normalization_cfg.get("eps", 1e-6),
    )

    scale_z_tmp = summarize_real_scale(X_tmp, tag=tag + "_zscore")

    pred_std, pred_phys, emb = predict_model_phys(
        model=model,
        X=X_tmp,
        y_mean=y_mean,
        y_std=y_std,
    )

    for j, name in enumerate(label_names):
        noise_control_results_m10.append({
            "center_offset_s": float(dt_center),
            "preprocessing": tag,
            "label": name,
            "pred": float(pred_phys[0, j]),
            "is_event_window": False,
        })

    noise_control_scales_m10.append(scale_raw_tmp)
    noise_control_scales_m10.append(scale_z_tmp)

noise_control_result_m10_df = pd.DataFrame(noise_control_results_m10)
noise_control_scale_m10_df = pd.concat(noise_control_scales_m10, ignore_index=True)

display(
    noise_control_result_m10_df.pivot_table(
        index="center_offset_s",
        columns="label",
        values="pred",
        aggfunc="first",
    )
)


noise_center_-1800s

noise_center_-1600s

noise_center_-1400s

noise_center_-1200s

noise_center_-1000s

noise_center_-800s

noise_center_-600s

noise_center_-400s

noise_center_+200s

noise_center_+300s


label,chi_eff,chirp_mass,total_mass
center_offset_s,,,
-1800.0,-0.049431,9.790863,25.793095
-1600.0,-0.158281,8.532259,22.392767
-1400.0,0.133105,9.009248,24.469991
-1200.0,-0.115271,9.029563,22.358120
-1000.0,-0.065795,8.320638,21.025085
-800.0,0.108259,8.193584,22.108118
-600.0,-0.082388,9.125532,24.276256
-400.0,-0.043953,8.605303,24.252927
200.0,-0.044559,7.782180,21.319814


## 13. Detector ablation

In [100]:
def run_detector_ablation_point(model, X_base, y_mean, y_std):
    variants = {}

    variants["HLV"] = X_base.copy()

    X_h1 = X_base.copy()
    X_h1[:, 1, :] = 0.0
    X_h1[:, 2, :] = 0.0
    variants["H1_only"] = X_h1

    X_hl = X_base.copy()
    X_hl[:, 2, :] = 0.0
    variants["H1_L1"] = X_hl

    X_hv = X_base.copy()
    X_hv[:, 1, :] = 0.0
    variants["H1_V1"] = X_hv

    rows = []

    for tag, X_var in variants.items():
        pred_std, pred_phys, emb = predict_model_phys(
            model=model,
            X=X_var,
            y_mean=y_mean,
            y_std=y_std,
        )

        for j, name in enumerate(label_names):
            rows.append({
                "variant": tag,
                "label": name,
                "pred": float(pred_phys[0, j]),
            })

    return pd.DataFrame(rows)

ablation_m10_df = run_detector_ablation_point(
    model=model,
    X_base=X_real,
    y_mean=y_mean,
    y_std=y_std,
)

display(
    ablation_m10_df.pivot_table(
        index="variant",
        columns="label",
        values="pred",
        aggfunc="first",
    )
)

label,chi_eff,chirp_mass,total_mass
variant,,,
H1_L1,0.085032,26.371180,61.408182
H1_V1,-0.281144,26.230900,60.982327
H1_only,-0.213415,24.011682,55.874545
HLV,-0.210869,25.376774,60.308495


M10-500k preserves the main transfer improvement observed with M10-small. The point estimate for GW170814 remains in a plausible BBH region and no longer shows the low-mass collapse observed with the original M08 real-data transfer test.

For the nominal event window, M10-500k predicts approximately:
- chirp mass: 25.38 Msun
- total mass: 60.31 Msun
- chi_eff: -0.211

The remaining diagnostics should be interpreted as transfer checks, not as a full Bayesian parameter-estimation analysis.

## 14. Event-vs-noise Summary

In [106]:
event_summary = pd.DataFrame({
    "window": ["GW170814_event"],
    "chirp_mass": [25.376774],
    "total_mass": [60.308495],
    "chi_eff": [-0.210869],
})

noise_summary = noise_control_result_m10_df.pivot_table(
    index="center_offset_s",
    columns="label",
    values="pred",
    aggfunc="first",
).reset_index()

display(event_summary)
display(noise_summary.describe())

,window,chirp_mass,total_mass,chi_eff
0,GW170814_event,25.376774,60.308495,-0.210869


label,center_offset_s,chi_eff,chirp_mass,total_mass
count,10.000000,10.000000,10.000000,10.000000
mean,-830.000000,-0.026761,8.650526,22.942396
std,714.998057,0.094671,0.593638,1.629556
min,-1800.000000,-0.158281,7.782180,21.025085
25%,-1350.000000,-0.078240,8.225348,21.597870
50%,-900.000000,-0.046995,8.568781,22.375443
75%,-450.000000,0.027038,9.024484,24.270424
max,300.000000,0.133105,9.790863,25.793095


In [105]:
noise_pivot = noise_control_result_m10_df.pivot_table(
    index="center_offset_s",
    columns="label",
    values="pred",
    aggfunc="first",
)

comparison_rows = []

for label in ["chirp_mass", "total_mass", "chi_eff"]:
    event_value = float(real_m10_point_df.loc[
        real_m10_point_df["label"] == label,
        "pred"
    ].iloc[0])

    noise_values = noise_pivot[label].values

    comparison_rows.append({
        "label": label,
        "event_pred": event_value,
        "noise_min": float(np.min(noise_values)),
        "noise_median": float(np.median(noise_values)),
        "noise_max": float(np.max(noise_values)),
        "event_minus_noise_median": float(event_value - np.median(noise_values)),
    })

event_vs_noise_df = pd.DataFrame(comparison_rows)
display(event_vs_noise_df)

,label,event_pred,noise_min,noise_median,noise_max,event_minus_noise_median
0,chirp_mass,25.376774,7.782180,8.568781,9.790863,16.807993
1,total_mass,60.308495,21.025085,22.375443,25.793095,37.933051
2,chi_eff,-0.210869,-0.158281,-0.046995,0.133105,-0.163874


M10-small corrige el fallo principal de transferencia real observado en M08.

Con evidencia:

- M08 colapsaba con V1: masas de ~3 y ~9.
- M10 HLV da una predicción plausible: chirp_mass ~28, total_mass ~67.
- M10 es estable frente a PSD window.
- M10 es estable frente a desplazamientos temporales razonables.
- M10 distingue evento de ruido off-source, porque las ventanas sin evento dan masas mucho menores.

Esto ya justifica escalar la línea M10.

## 15. M10 real-event diagnostic conclusion

The M10 input-zscore model substantially improves the real-event behaviour relative to the original M08 baseline. For GW170814, the HLV prediction moves from the unphysical low-mass collapse observed with M08 to a plausible BBH region.

The M10 prediction is stable under reasonable PSD-window choices and under center-time shifts of ±1.5 s. Detector ablation no longer shows a pathological collapse when V1 is included, indicating that the per-sample, per-detector input normalization has mitigated the detector-dependent scale mismatch observed in the real GWOSC strain.

Off-source real-noise controls further show that M10 does not generically predict high-mass BBH parameters for arbitrary real strain windows. The GW170814 event window stands out clearly from nearby off-source noise windows, especially in chirp mass and total mass.

These results support adopting M10-500k as the normalized-input baseline for the real-event transfer study. The conformal intervals are calibrated with the final M10-500k calibration split and should be reported as split-conformal prediction intervals for the CNN estimator, not as astrophysical posterior credible intervals.


__________________________________

## Load predictions M10-500k and Mondrian selection

In [87]:
from pathlib import Path
import numpy as np
import pandas as pd

from src.conformal.pipeline import run_mondrian_regression

DATASET_ID = "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000"

PRED_500K_PATH = DATA_ROOT / "results" / DATASET_ID / "m10_inputzscore_500k_cal_test_predictions_embeddings.npz"

# fallback si tu archivo está directamente en results/
if not PRED_500K_PATH.exists():
    PRED_500K_PATH = DATA_ROOT / "results" / "m10_inputzscore_500k_cal_test_predictions_embeddings.npz"

MONDRIAN_DIR = DATA_ROOT / "results" / "mondrian_M10_final_baseline"

# Si cambias el nombre recomendado, usa:
# MONDRIAN_DIR = DATA_ROOT / "results" / DATASET_ID / "mondrian_m10_inputzscore_500k_final"

SELECTION_PATH = MONDRIAN_DIR / "selected_configurations.csv"

print("PRED_500K_PATH:", PRED_500K_PATH, PRED_500K_PATH.exists())
print("SELECTION_PATH:", SELECTION_PATH, SELECTION_PATH.exists())

if not PRED_500K_PATH.exists():
    raise FileNotFoundError(PRED_500K_PATH)

if not SELECTION_PATH.exists():
    raise FileNotFoundError(SELECTION_PATH)

pred_data = np.load(PRED_500K_PATH, allow_pickle=True)
selection_df = pd.read_csv(SELECTION_PATH)

display(selection_df)

PRED_500K_PATH: /data/vserrano/cbc_pe_data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/m10_inputzscore_500k_cal_test_predictions_embeddings.npz True
SELECTION_PATH: /data/vserrano/cbc_pe_data/results/mondrian_M10_final_baseline/selected_configurations.csv True


,taxonomy_mode,interval_mode,n_bins,label,label_index,global_coverage,global_miscoverage,global_undercoverage_pvalue,global_mean_width_std,global_median_width_std,...,global_2sigma_low,global_2sigma_high,global_3sigma_low,global_3sigma_high,global_within_2sigma,min_width_for_label,width_limit,relative_width_excess,selection_policy,final_policy
0,difficulty,asymmetric,4,chirp_mass,0,0.900533,0.099467,0.623708,0.883767,0.714333,...,0.896536,0.903464,0.894804,0.905196,True,11.731975,11.966614,0.003718,conservative_zero_under_bins_2sigma,conservative
1,difficulty,symmetric,4,chirp_mass,0,0.901167,0.098833,0.752327,0.886183,0.711688,...,0.896536,0.903464,0.894804,0.905196,True,11.731975,11.966614,0.000000,efficient_local_validity_tolerant,efficient
2,prediction,asymmetric,6,total_mass,1,0.899533,0.100467,0.396584,0.827581,0.756104,...,0.896536,0.903464,0.894804,0.905196,True,25.844899,26.361797,0.013706,conservative_zero_under_bins_2sigma,conservative
3,difficulty,symmetric,4,total_mass,1,0.900567,0.099433,0.631003,0.840208,0.745881,...,0.896536,0.903464,0.894804,0.905196,True,25.844899,26.361797,0.000000,efficient_local_validity_tolerant,efficient
4,prediction,symmetric,8,chi_eff,2,0.900267,0.099733,0.563986,1.336248,1.321660,...,0.896536,0.903464,0.894804,0.905196,True,0.579787,0.591383,0.004957,conservative_zero_under_bins_2sigma,conservative
5,difficulty,asymmetric,12,chi_eff,2,0.899033,0.100967,0.291065,1.318723,1.231796,...,0.896536,0.903464,0.894804,0.905196,True,0.543044,0.553905,0.000000,efficient_local_validity_tolerant,efficient


In [88]:
pred_cal = pred_data["pred_cal"].astype(np.float64)
y_cal = pred_data["y_cal"].astype(np.float64)
emb_cal = pred_data["emb_cal"].astype(np.float64)

label_names_from_pred = [str(x) for x in pred_data["label_names"].tolist()]

print("pred_cal:", pred_cal.shape)
print("y_cal:", y_cal.shape)
print("emb_cal:", emb_cal.shape)
print("label_names:", label_names_from_pred)

assert label_names_from_pred == label_names

pred_cal: (30000, 3)
y_cal: (30000, 3)
emb_cal: (30000, 64)
label_names: ['chirp_mass', 'total_mass', 'chi_eff']


In [89]:
print("emb_cal type:", type(emb_cal))
print("emb_cal shape:", None if emb_cal is None else emb_cal.shape)

print("emb_real type:", type(emb_real))
print("emb_real shape:", None if emb_real is None else emb_real.shape)

print("pred_real_std:", pred_real_std.shape)
print("pred_real_phys:", pred_real_phys.shape)

emb_cal type: <class 'numpy.ndarray'>
emb_cal shape: (30000, 64)
emb_real type: <class 'numpy.ndarray'>
emb_real shape: (1, 64)
pred_real_std: (1, 3)
pred_real_phys: (1, 3)


In [92]:
CONFIDENCE_LEVEL = 0.90
N_NEIGHBORS = 5
MIN_SAMPLES_PER_BIN = 20

def apply_selected_mondrian_to_real(
    final_policy,
    pred_real_std,
    emb_real,
    pred_real_phys,
):
    rows = []

    # y_test dummy sólo para que la función pueda construir result/metrics.
    # No interpretamos cobertura aquí.
    y_dummy = pred_real_std.copy()

    for _, sel in selection_df[selection_df["final_policy"] == final_policy].iterrows():
        label = sel["label"]
        j = label_names.index(label)

        taxonomy_mode = sel["taxonomy_mode"]
        interval_mode = sel["interval_mode"]
        n_bins = int(sel["n_bins"])

        kwargs = dict(
            pred_cal=pred_cal,
            pred_test=pred_real_std,
            y_cal=y_cal,
            y_test=y_dummy,
            n_bins=n_bins,
            confidence_level=CONFIDENCE_LEVEL,
            apply_jitter=True,
            interval_mode=interval_mode,
            taxonomy_mode=taxonomy_mode,
            min_samples_per_bin=MIN_SAMPLES_PER_BIN,
            tolerance_sigmas=(1, 2, 3),
        )

        if taxonomy_mode == "difficulty":
            if emb_cal is None:
                raise ValueError("Selected difficulty taxonomy requires emb_cal, but emb_cal is None.")
            if emb_real is None:
                raise ValueError("Selected difficulty taxonomy requires emb_real, but emb_real is None.")

            kwargs.update(
                cal_embedding=emb_cal,
                target_embedding=emb_real,
                n_neighbors=N_NEIGHBORS,
            )

        result = run_mondrian_regression(**kwargs)

        lower_std = float(result.lower[0, j])
        upper_std = float(result.upper[0, j])

        lower_phys = lower_std * y_std[j] + y_mean[j]
        upper_phys = upper_std * y_std[j] + y_mean[j]

        rows.append({
            "event": event_name,
            "label": label,
            "final_policy": final_policy,
            "taxonomy_mode": taxonomy_mode,
            "interval_mode": interval_mode,
            "n_bins": n_bins,
            "pred_std": float(pred_real_std[0, j]),
            "pred_phys": float(pred_real_phys[0, j]),
            "lower_phys": float(lower_phys),
            "upper_phys": float(upper_phys),
            "width_phys": float(upper_phys - lower_phys),
        })

    return pd.DataFrame(rows)


real_m10_500k_conservative_df = apply_selected_mondrian_to_real(
    final_policy="conservative",
    pred_real_std=pred_real_std,
    emb_real=emb_real,
    pred_real_phys=pred_real_phys,
)

display(real_m10_500k_conservative_df)

,event,label,final_policy,taxonomy_mode,interval_mode,n_bins,pred_std,pred_phys,lower_phys,upper_phys,width_phys
0,GW170814,chirp_mass,conservative,difficulty,asymmetric,4,-0.732548,25.376774,13.420476,36.086134,22.665657
1,GW170814,total_mass,conservative,prediction,asymmetric,6,-1.000442,60.308495,47.216019,68.521850,21.305831
2,GW170814,chi_eff,conservative,prediction,symmetric,8,-0.480637,-0.210869,-0.540139,0.118401,0.658540


In [91]:
real_m10_500k_efficient_df = apply_selected_mondrian_to_real(
    final_policy="efficient",
    pred_real_std=pred_real_std,
    emb_real=emb_real,
    pred_real_phys=pred_real_phys,
)

display(real_m10_500k_efficient_df)

,event,label,final_policy,taxonomy_mode,interval_mode,n_bins,pred_std,pred_phys,lower_phys,upper_phys,width_phys
0,GW170814,chirp_mass,efficient,difficulty,symmetric,4,-0.732548,25.376774,13.989404,36.764145,22.774741
1,GW170814,total_mass,efficient,difficulty,symmetric,4,-1.000442,60.308495,40.861863,79.755127,38.893264
2,GW170814,chi_eff,efficient,difficulty,asymmetric,12,-0.480637,-0.210869,-0.692167,0.215658,0.907825


The M10-500k model predicts a plausible BBH region for GW170814. The selected conservative Mondrian calibration assigns the event to a high-difficulty chirp-mass bin, leading to a wider conformal interval for M. This is consistent with the intended behaviour of the difficulty-based Mondrian taxonomy: locally harder regions receive wider intervals.

These intervals are split-conformal prediction intervals for the CNN estimator under the synthetic calibration distribution. They should not be interpreted as Bayesian posterior credible intervals.


## Final M10-500k real-event conclusion

The M10-500k input-zscore model preserves the real-data robustness observed in the M10-small diagnostic run. For GW170814, the model predicts a plausible BBH region and avoids the low-mass collapse observed with the original non-normalized M08 transfer test.

Using the final conservative Mondrian calibration, the predicted parameters are:

- chirp mass: 25.38 Msun, conformal interval [13.42, 36.09] Msun
- total mass: 60.31 Msun, conformal interval [47.22, 68.52] Msun
- chi_eff: -0.211, conformal interval [-0.540, 0.118]

The chirp-mass interval is comparatively wide because the event falls into a high-difficulty Mondrian bin. This is expected behaviour for the difficulty-based taxonomy and indicates that the conformal system assigns larger uncertainty to locally harder regions of the learned embedding.

These intervals are split-conformal prediction intervals calibrated on synthetic BBH data. They are not Bayesian posterior credible intervals and should be interpreted as uncertainty intervals for the trained CNN estimator under the synthetic calibration distribution.